In [1]:
import numpy as np
import scipy as sp
from typing import Callable, Union
import pandas as pd
import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', True)
import os
import pickle
import logging
import time
import sys
import matplotlib.pyplot as plt
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
lgcg_path = os.path.abspath(os.path.join('../nlgcg'))
if lgcg_path not in sys.path:
    sys.path.append(lgcg_path)
from lib.measure import Measure
from lib.ssn import SSN
from nlgcg import NLGCG

# Heat Equation

## Generate Data and Define Functions

In [ ]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [ ]:
# Omega = np.array([[0,1], [0,1]])
# alpha = 1e-1
# observation_resolution = 4
# std_factor = 0.1
# true_sources = np.array([[0.28, 0.71], [0.51,0.27], [0.71,0.53]])
# true_weights = np.array([1,-0.7, 0.8])
# true_measure = Measure(support=true_sources, coefficients=true_weights)

In [ ]:
Omega = np.array([[0,1], [0,1]])
alpha = 2e-1
observation_resolution = 10
std_factor = 0.02
np.random.seed(49)
true_sources = np.random.rand(20, 2)*0.9
true_weights = np.random.rand(20) * 2 - 1
true_measure = Measure(support=true_sources, coefficients=true_weights)

In [ ]:
observations = (np.array(np.meshgrid(
                    *(
                        np.linspace(bound[0], bound[1], observation_resolution+2)
                        for bound in Omega
                    ))
            ).reshape(len(Omega), -1).T)
observations = np.array([obs for obs in observations if all(obs!=0) and all(obs!=1)])

In [ ]:
@jax.jit
def kernel(omega: np.ndarray):
    outer_factor = np.sqrt(std_factor*np.pi)**Omega.shape[0]
    norms = -jnp.square(jnp.linalg.norm(omega-observations, axis=1))/std_factor # (len(x),)
    exponentiated = jnp.exp(norms)/outer_factor # (len(x),)
    return exponentiated

grad_kernel = jax.jit(jax.jacobian(kernel))
hess_kernel = jax.jit(jax.hessian(kernel))
_ = grad_kernel(true_sources[0])
_ = hess_kernel(true_sources[0])

kernel = jax.vmap(kernel)
grad_kernel = jax.vmap(grad_kernel)
hess_kernel = jax.vmap(hess_kernel)

In [ ]:
target = true_measure.duality_pairing(kernel)

@jax.jit
def g(w: np.ndarray) -> float:
    return alpha * jnp.linalg.norm(w, ord=1)
# grad_g = jax.jit(jax.grad(g))
# _ = grad_g(jnp.ones(1))

@jax.jit
def f(y: np.ndarray) -> float:
    return 0.5 * jnp.sum((y - target)**2)
f_grad = jax.jit(jax.grad(f))
_ = f_grad(jnp.zeros(target.shape[0]))

j = lambda u: f(u.duality_pairing(kernel)) + g(u.coefficients)

In [ ]:
def p(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: kernel(omega) @ inner

def grad_P(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: np.tensordot(grad_kernel(omega), inner, axes=([1,0]))

def hess_P(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: np.tensordot(hess_kernel(omega), inner, axes=([1,0]))

In [ ]:
@jax.jit
def j_N(input: np.ndarray):
    input = input.reshape(-1, Omega.shape[0]+1)
    weights = input[:,0]
    omega = input[:,1:]
    return f(kernel(omega).T@weights) + g(weights)

grad_j_N = jax.jit(jax.grad(j_N))
hess_j_N = jax.jit(jax.hessian(j_N))
_ = grad_j_N(np.ones((2, Omega.shape[0]+1)).flatten())
_ = hess_j_N(np.ones((2, Omega.shape[0]+1)).flatten())

## Experiments

In [ ]:
exp = NLGCG(target=target, 
           kernel=kernel, 
           g=g, 
           j=j,
           j_N=j_N,
           p=p,
           grad_P=grad_P,
           hess_P=hess_P,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=4,
           dual_variable_goodness=0.3,
           armijo_constant=0.1
           )

In [ ]:
u, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.nlgcg(tol=1e-12, max_radius=0.1, mode="stochastic")

In [ ]:
np.array(times)

In [ ]:
print(u)

In [ ]:
hesses = exp.hess_P(u)(u.support)
grads = exp.grad_P(u)(u.support)

In [ ]:
np.linalg.eigvals(hesses[5])

In [ ]:
u.to_matrix().shape

In [ ]:
hess = hess_j_N(u.to_matrix().flatten())
np.linalg.eigvals(hess)

## Plots

In [ ]:
p_u = p(u)
a = np.arange(0,1,0.01)
x, y = np.meshgrid(a,a)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = np.abs(p_u(points)).reshape((100,100))

plt.contourf(x, y, vals, levels=100);
for omega in u.support:
    plt.plot(omega[0], omega[1], "o", c="black", markersize=5);
plt.colorbar();

In [ ]:
p_u = p(u)
P = lambda x: np.abs(p_u(x))
a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
points = np.array(list(zip(B.flatten(), D.flatten())))
vals = P(points).reshape((100,100))

plt.contourf(B, D, vals, levels=100);
plt.colorbar();
plt.plot(Omega[0][0]-1, Omega[0][1]-1, "P", c="black", markersize=8, label="True sources");
plt.plot(Omega[0][0]-1, Omega[0][1]-1, "o", c="black", label="Optimal support");
plt.xlim(Omega[0][0], Omega[0][1]);
plt.ylim(Omega[1][0], Omega[1][1]);
for i, x in enumerate(true_sources):
    if true_weights[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(true_weights[i]) * 12 + 4
    plt.plot([x[0]], [x[1]], "P", alpha=0.5, c=color, markersize=size);
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i]) * 10 + 2
    plt.plot([x[0]], [x[1]], "o", c=color, markersize=size);
plt.legend();

In [ ]:
# Plot the measured heat distribution
def heat(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    if len(x.shape) == 1:
        x = x.reshape(1, -1) 
    weighted_heat = np.zeros(x.shape[0]) # (len(x),)
    outer_factor = np.sqrt(std_factor*np.pi)**Omega.shape[0]
    for point, weight in zip(true_sources, true_weights):
        diff = point-x # (len(x), Omega.shape[0])
        norms = -np.square(np.linalg.norm(diff, axis=1))/std_factor # (len(x),)
        exponentiated = np.exp(norms) # (len(x),)
        weighted_heat += exponentiated * weight # (len(x),)
    result = weighted_heat/outer_factor # shape=(len(x),)
    return result

a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
x = np.array(list(zip(B.flatten(), D.flatten())))
true_vals = heat(x).reshape((100,100))

plt.contourf(B, D, true_vals, levels=100);
plt.colorbar();
plt.xlim(Omega[0][0], Omega[0][1]);
plt.ylim(Omega[1][0], Omega[1][1]);
for i, x in enumerate(true_sources):
    if true_weights[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(true_weights[i]) * 12 + 4
    plt.plot([x[0]], [x[1]], "P", c=color, markersize=size);

In [ ]:
def predicted_heat(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    if len(x.shape) == 1:
        x = x.reshape(1, -1) 
    weighted_heat = np.zeros(x.shape[0]) # (len(x),)
    outer_factor = np.sqrt(std_factor*np.pi)**Omega.shape[0]
    for point, weight in zip(u.support, u.coefficients):
        diff = point-x # (len(x), Omega.shape[0])
        norms = -np.square(np.linalg.norm(diff, axis=1))/std_factor # (len(x),)
        exponentiated = np.exp(norms) # (len(x),)
        weighted_heat += exponentiated * weight # (len(x),)
    result = weighted_heat/outer_factor # shape=(len(x),)
    return result

a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
x = np.array(list(zip(B.flatten(), D.flatten())))
pred_vals = predicted_heat(x).reshape((100,100))
error = true_vals - pred_vals

print(f"L2 error: {np.linalg.norm(error)/np.sqrt(len(x)):.3E}, Linf error: {np.max(np.abs(error)):.3E}")

plt.contourf(B, D, np.abs(error), levels=100);
plt.colorbar();
plt.plot(Omega[0][0]-1, Omega[0][1]-1, "P", c="black", markersize=8, label="True sources");
plt.plot(Omega[0][0]-1, Omega[0][1]-1, "o", c="black", label="Optimal support");
plt.xlim(Omega[0][0], Omega[0][1]);
plt.ylim(Omega[1][0], Omega[1][1]);
for i, x in enumerate(true_sources):
    if true_weights[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(true_weights[i]) * 12 + 4
    plt.plot([x[0]], [x[1]], "P", alpha=0.5, c=color, markersize=size);
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i]) * 10 + 2
    plt.plot([x[0]], [x[1]], "o", c=color, markersize=size);
plt.legend();

# Signal Processing

## Generate Data and Define Functions

In [ ]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [ ]:
observation_resolution = 120
Omega = np.array([[0,observation_resolution//2]])
alpha = 1e-1
true_sources = np.array([[3.125], [7], [np.sqrt(179)]])
true_weights = np.array([-1, 0.7, 0.5])
true_measure = Measure(support=true_sources, coefficients=true_weights)

In [ ]:
# alpha = 1e-0
# source_number = 20
# observation_resolution = 100
# Omega = np.array([[0,observation_resolution//2]])
# np.random.seed(49)
# true_sources = np.array(np.random.rand(source_number)*50)
# true_weights = np.random.rand(source_number) * 2 - 1
# true_measure = Measure(support=true_sources, coefficients=true_weights)

In [ ]:
observations = np.arange(0,1,1/observation_resolution)

In [ ]:
@jax.jit
def kernel(omega: np.ndarray):
    return jnp.sin(2*np.pi*omega*observations).flatten()

grad_kernel = jax.jit(jax.jacobian(kernel))
hess_kernel = jax.jit(jax.hessian(kernel))
_ = grad_kernel(true_sources[0])
_ = hess_kernel(true_sources[0])

kernel = jax.vmap(kernel)
grad_kernel = jax.vmap(grad_kernel)
hess_kernel = jax.vmap(hess_kernel)

In [ ]:
target = true_measure.duality_pairing(kernel)

@jax.jit
def g(w: np.ndarray) -> float:
    return alpha * jnp.linalg.norm(w, ord=1)
# grad_g = jax.jit(jax.grad(g))
# _ = grad_g(jnp.ones(1))

@jax.jit
def f(y: np.ndarray) -> float:
    return 0.5 * jnp.sum((y - target)**2)
f_grad = jax.jit(jax.grad(f))
_ = f_grad(jnp.zeros(target.shape[0]))

j = lambda u: f(u.duality_pairing(kernel)) + g(u.coefficients)

In [ ]:
def p(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: kernel(omega) @ inner

def grad_P(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: np.tensordot(grad_kernel(omega), inner, axes=([1,0]))

def hess_P(u):
    inner = -f_grad(u.duality_pairing(kernel, len(target)))
    return lambda omega: np.tensordot(hess_kernel(omega), inner, axes=([1,0]))

In [ ]:
@jax.jit
def j_N(input: np.ndarray):
    input = input.reshape(-1, Omega.shape[0]+1)
    weights = input[:,0]
    omega = input[:,1:]
    return f(kernel(omega).T@weights) + g(weights)

grad_j_N = jax.jit(jax.grad(j_N))
hess_j_N = jax.jit(jax.hessian(j_N))
_ = grad_j_N(np.ones((2, Omega.shape[0]+1)).flatten())
_ = hess_j_N(np.ones((2, Omega.shape[0]+1)).flatten())

## Experiments

In [ ]:
exp = NLGCG(target=target, 
           kernel=kernel, 
           g=g, 
           j=j,
           j_N=j_N,
           p=p,
           grad_P=grad_P,
           hess_P=hess_P,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=1000,
           dual_variable_goodness=0.5,
           armijo_constant=0.1
           )

In [ ]:
u, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.nlgcg(tol=1e-12, max_radius=1, mode="stochastic")

In [ ]:
np.array(times)

In [ ]:
dropped_tot

In [ ]:
print(u)

## Plots

In [ ]:
a = np.arange(Omega[0][0],Omega[0][1],0.001)
p_u = p(u)
vals = p_u(a)
plt.plot(a,vals);
plt.axvline(x=-1, linestyle="-", c="r", label="Support");
plt.axvline(x=-1, linestyle="-", c="g", label="Truth");
for i, pos in enumerate(u.support):
    ymax = 0.5+(u.coefficients[i]*alpha*0.25+0.05*np.sign(u.coefficients[i]))
    plt.axvline(x=pos, ymin=0.5,ymax=ymax, linestyle="-", c="r");
for point, weight in  zip(true_sources, true_weights):
    ymax = 0.5+(weight*alpha*0.25+0.05*np.sign(weight))
    plt.axvline(x=point, ymin=0.5,ymax=ymax,alpha=0.5, linestyle="-", c="g");
plt.xlabel("Frequency");
plt.ylim(-alpha*1.1,alpha*1.1);
plt.xlim(0, 50);
plt.legend();

In [ ]:
# Plot the measured signal
def signal(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    weighted_signal = 0
    for point, weight in zip(true_sources, true_weights):
        weighted_signal += weight*np.sin(2*np.pi*point*x).flatten()
    return weighted_signal

a = np.arange(0,1,0.001)
true_signal = signal(a)
plt.plot(a, true_signal);
plt.xlabel("Time");

In [ ]:
# Plot the measured signal
def signal(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    weighted_signal = 0
    for point, weight in zip(u.support, u.coefficients):
        weighted_signal += weight*np.sin(2*np.pi*point*x).flatten()
    return weighted_signal

a = np.arange(0,1,0.001)
predicted_signal = signal(a)
error = true_signal - predicted_signal
print(f"L2 error: {np.linalg.norm(error)/np.sqrt(len(a))}, Linf error: {np.max(np.abs(error))}")
plt.plot(a, np.abs(true_signal-predicted_signal));
plt.xlabel("Time");

# Function Approximation (Gaussian Shallow NN)

## Generate Data and Define Functions

In [4]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [5]:
# sigma_space = np.array([[0,1]])
# omega_space = np.array([[0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 100
# endpoint = True
# true_sigma = np.array([0.06, 0.05, 0.04])
# true_omega = np.array([[0.28], [0.51], [0.71]])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.array([1, -0.7, 0.8])
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [6]:
# sigma_space = np.array([[0,1]])
# omega_space = np.array([[0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 100
# endpoint = True
# np.random.seed(49)
# true_sigma = np.random.rand(10)*0.1
# true_omega = (np.random.rand(10, 1)-np.array([0.5]))*0.9+np.array([0.5])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.random.rand(10) * 2 - 1
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [7]:
# sigma_space = np.array([[0,1]])
# omega_space = np.array([[-1,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 100
# endpoint = True

# true_function = lambda x: sp.special.expit(100*x).flatten()
# method = "function"

In [8]:
# sigma_space = np.array([0,1])
# omega_space = np.array([[0,1], [0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 25
# endpoint = True
# true_sigma = np.array([0.06, 0.05, 0.04])
# true_omega = np.array([[0.28, 0.71], [0.51,0.27], [0.71,0.53]])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.array([1, -0.7, 0.8])
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [9]:
# sigma_space = np.array([0,1])
# omega_space = np.array([[0,1], [0,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 3e-2
# observation_resolution = 25
# endpoint = True
# np.random.seed(49)
# true_sigma = np.random.rand(20)*0.2
# true_omega = (np.random.rand(20, 2)-np.array([0.5,0.5]))*0.9+np.array([0.5,0.5])
# true_sources = np.hstack((true_sigma.reshape(-1,1), true_omega))
# true_weights = np.random.rand(20) * 2 - 1
# true_measure = Measure(support=true_sources, coefficients=true_weights)
# method = "measure"

In [10]:
# sigma_space = np.array([0,1])
# omega_space = np.array([[-1,1], [-1,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 1e-3
# observation_resolution = 10
# endpoint = True

# def my_funct(x: np.ndarray, c: np.ndarray, k: float, r: float):
#     return np.tanh(k*(r-np.linalg.norm(x-c,axis=1)))+1

# c_1 = np.array([0.3,0.3])
# k_1 = 4
# r_1 = 0.3
# c_2 = -c_1
# k_2 = 12
# r_2 = 0.15
# true_function = lambda x: my_funct(x, c_1, k_1, r_1) + my_funct(x, c_2, k_2, r_2)
# method = "function"

In [11]:
# sigma_space = np.array([0,1])
# omega_space = np.array([[-1,1], [-1,1]])
# Omega = np.vstack((sigma_space, omega_space))
# d = omega_space.shape[0]
# variance_exponent = 0.1
# alpha = 5e-3
# observation_resolution = 10
# endpoint = False

# true_function = lambda x: np.minimum(1-np.abs(x[:,0]), 1-np.abs(x[:,1]))
# method = "function"

In [12]:
sigma_space = np.array([0,1])
omega_space = np.array([[-1,1], [-1,1], [-1,1], [-1,1]])
Omega = np.vstack((sigma_space, omega_space))
d = omega_space.shape[0]
variance_exponent = 0
alpha = 5e-8
observation_resolution = 5
endpoint = False

def my_funct(x: np.ndarray):
    to_return = np.ones(x.shape[0])
    for i in range(x.shape[1]):
        to_return *= np.sin(np.pi*x[:,i])
    return to_return

true_function = lambda x: my_funct(x)
method = "function"

In [13]:
observations = (np.array(np.meshgrid(
                    *(
                        np.linspace(bound[0]+1e-5, bound[1]-1e-5, observation_resolution, endpoint=endpoint)
                        for bound in omega_space
                    ))
            ).reshape(len(omega_space), -1).T)
observations = np.array([obs for obs in observations])# if all(obs!=0) and all(obs!=1)])

In [14]:
@jax.jit
def singleton_kernel(omega: np.ndarray):
    sigma = omega[0]
    x = omega[1:]
    inner = -(jnp.linalg.norm(x - observations,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**(variance_exponent))/(np.sqrt(2*np.pi)**d)
    return outer

grad_kernel = jax.jit(jax.jacobian(singleton_kernel))
hess_kernel = jax.jit(jax.hessian(singleton_kernel))
_ = grad_kernel(np.ones(len(Omega)))
_ = hess_kernel(np.ones(len(Omega)))

kernel = jax.vmap(singleton_kernel)
grad_kernel = jax.vmap(grad_kernel)
hess_kernel = jax.vmap(hess_kernel)

INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


In [15]:
if method == "measure":
    target = true_measure.duality_pairing(kernel)
elif method == "function":
    target = true_function(observations)

In [16]:
@jax.jit
def g(w: np.ndarray) -> float:
    return alpha * jnp.linalg.norm(w, ord=1)
grad_g = jax.jit(jax.grad(g))
_ = grad_g(jnp.ones(1))

@jax.jit
def f(y: np.ndarray) -> float:
    return 0.5 * jnp.sum((y - target)**2)
grad_f = jax.jit(jax.grad(f))
hess_f = jax.jit(jax.hessian(f))
_ = grad_f(jnp.zeros(target.shape[0]))
_ = hess_f(jnp.zeros(target.shape[0]))

j = lambda u, c: f(u.duality_pairing(kernel)+c*np.ones(target.shape)) + g(u.coefficients)

In [17]:
# def p(u, c):
#     inner = -grad_f(u.duality_pairing(kernel, len(target))+c*np.ones(target.shape))
#     return lambda omega: kernel(omega) @ inner

# def grad_p(u, c):
#     inner = -grad_f(u.duality_pairing(kernel, len(target))+c*np.ones(target.shape))
#     return lambda omega: np.tensordot(grad_kernel(omega), inner, axes=([1,0]))

# def hess_p(u, c):
#     inner = -grad_f(u.duality_pairing(kernel, len(target))+c*np.ones(target.shape))
#     return lambda omega: np.tensordot(hess_kernel(omega), inner, axes=([1,0]))

In [18]:
@jax.jit
def p_raw(parameters: np.ndarray, c: float, omega: np.ndarray):
    coefficients = parameters[:,0]
    support = parameters[:,1:]
    Ku = jnp.tensordot(kernel(support), coefficients, axes=([0], [0]))
    constant_term = c*jnp.ones(len(target))
    return singleton_kernel(omega) @ -grad_f(Ku+constant_term)

grad_p_raw = jax.jit(jax.grad(p_raw, argnums=2))
hess_p_raw = jax.jit(jax.hessian(p_raw, argnums=2))

p_raw = jax.jit(jax.vmap(p_raw, in_axes=(None, None, 0), out_axes=0))
grad_p_raw = jax.jit(jax.vmap(grad_p_raw, in_axes=(None, None, 0), out_axes=0))
hess_p_raw = jax.jit(jax.vmap(hess_p_raw, in_axes=(None, None, 0), out_axes=0))

p = lambda u, c: lambda omega: p_raw(u.to_matrix(len(Omega)), c, omega)
grad_p = lambda u, c: lambda omega: grad_p_raw(u.to_matrix(len(Omega)), c, omega)
hess_p = lambda u, c: lambda omega: hess_p_raw(u.to_matrix(len(Omega)), c, omega)

In [19]:
# Parameterized versions of f and j
@jax.jit
def f_N(raw_input: np.ndarray) -> float:
    constant = raw_input[-1]
    input = raw_input[:-1].reshape(-1, d+2)
    weights = input[:,0]
    omega = input[:,1:]
    return f(kernel(omega).T@weights+constant*jnp.ones(target.shape))

@jax.jit
def j_N(raw_input: np.ndarray) -> float:
    input = raw_input[:-1].reshape(-1, d+2)
    weights = input[:,0]
    return f_N(raw_input) + g(weights)

grad_f_N = jax.jit(jax.grad(f_N))
hess_f_N = jax.jit(jax.hessian(f_N))
grad_j_N = jax.jit(jax.grad(j_N))
hess_j_N = jax.jit(jax.hessian(j_N))

## Alternative Approach

In [ ]:
peaks, _ = sp.signal.find_peaks(target, prominence=0)

In [ ]:
def gaussian(x, sigma, x_0, coefficient, constant):
    return coefficient * np.exp(-(x-x_0)**2 / (2*sigma**2)) + constant

In [ ]:
plt.plot(target)
for i, peak in enumerate(peaks):
    interval = np.array(list(range(_["left_bases"][i], _["right_bases"][i])))
    popt, pcov = sp.optimize.curve_fit(gaussian, interval, target[interval],p0=[1,peak,1,0])
    plt.plot(interval, gaussian(interval, *popt), color="green")

## SSN Test

In [2]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [ ]:
# data_matrix_raw = pd.read_csv("data/gisette_scale", sep=" ").values
# data_matrix_raw.shape

In [ ]:
# data_matrix = np.zeros((5999, 5000))
# target_matrix = np.zeros(5999)
# for i, row in enumerate(data_matrix_raw):
#     if i % 100 == 0:
#         print(i)
#     target_matrix[i] = float(row[0])
#     for val in row[1:]:
#         try:
#             raw_ind, raw_val = val.split(":")
#             ind = int(raw_ind)
#             val = float(raw_val)
#             data_matrix[i][ind-1] = val
#         except AttributeError:
#             pass

In [ ]:
# with open('data/data_matrix.pkl', 'wb') as f:
#     pickle.dump(data_matrix, f)
# with open('data/target_matrix.pkl', 'wb') as f:
#     pickle.dump(target_matrix, f)

In [3]:
with open('data/data_matrix.pkl', 'rb') as f:
    data_matrix = pickle.load(f)
    data_matrix = data_matrix[:,:1000]
with open('data/target_matrix.pkl', 'rb') as f:
    target_matrix = pickle.load(f)

In [4]:
L = (np.linalg.norm(data_matrix)**2)/(4*data_matrix.shape[0])

In [5]:
alpha = 0.002

@jax.jit
def f_N(x: np.ndarray) -> float:
    inner = jnp.multiply(-target_matrix, jnp.matmul(data_matrix, x))
    outer = jnp.log(1+jnp.exp(inner))
    return jnp.mean(outer)

@jax.jit
def g(w: np.ndarray) -> float:
    return alpha * jnp.linalg.norm(w, ord=1)
grad_g = jax.jit(jax.grad(g))
_ = grad_g(jnp.ones(1))

@jax.jit
def j_N(x: np.ndarray) -> float:
    return f_N(x) + g(x)

grad_f_N = jax.jit(jax.grad(f_N))
# grad_f_N = lambda x: np.array(grad_f_N_(x))
hess_f_N = jax.jit(jax.hessian(f_N))
# hess_f_N = lambda x: np.array(hess_f_N_(x))
grad_j_N = jax.jit(jax.grad(j_N))
# grad_j_N = lambda x: np.array(grad_j_N_(x))
hess_j_N = jax.jit(jax.hessian(j_N))
# hess_j_N = lambda x: np.array(hess_j_N_(x))

INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


In [28]:
x = np.random.random(5000)*0.001
h = np.random.random(5000)*0.001

In [11]:
y, vjp_fun = jax.vjp(j_N, x)

In [19]:
%timeit grad_j_N(x)

19.9 ms ± 496 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [23]:
np.linalg.norm(y-j_N(x))

0.0

In [30]:
s = 0.000001
diff = (j_N(x+s*h)-j_N(x))/s
np.linalg.norm(vjp_fun(1.0)@h-diff)

1.9877020263070477e-07

In [22]:
%timeit j_N(x)

11.1 ms ± 112 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
%timeit jax.vjp(j_N, x)

53.6 ms ± 911 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [17]:
%timeit vjp_fun(1.0)

10.2 ms ± 95.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [35]:
%timeit np.array(jax.jvp(grad_j_N, (x,), (h,))[1])

43.4 ms ± 1.39 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [36]:
%timeit np.array(hess_j_N(x)@h)

7.33 s ± 124 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [33]:
def H(
    normal_map_vector: np.ndarray,
    prox_q: np.ndarray,
    tau: float,
    lbda: float = 1,
) -> float:
    return (
        j_N(prox_q) + tau * lbda * 0.5 * np.linalg.norm(normal_map_vector) ** 2
    )

def normal_map(
    q: np.ndarray, prox_q: np.ndarray, lbda: float = 1
) -> np.ndarray:
    grad_f_prox_q = grad_f_N(prox_q)
    return grad_f_prox_q + (q - prox_q) / lbda

def prox_and_grad(q: np.ndarray, lbda: float = 1) -> tuple:
    # prox_val = np.zeros(q.shape)
    prox_grad = np.zeros(q.shape)
    prox_val = jnp.multiply(jnp.sign(q), jnp.maximum(0, jnp.abs(q) - lbda*alpha))
    indices = jnp.where(prox_val != 0)[0]
    prox_grad[indices] = 1
    # for i, val in enumerate(q):
    #     if i % (Omega.shape[0] + 1) or i == (len(q) - 1):
    #         # Not coefficient: not regularized
    #         prox_val[i] = val
    #         prox_grad[i] = 1
    #     else:
    #         if np.abs(val) > alpha * lbda:
    #             prox_val[i] = val - lbda * alpha * np.sign(val)
    #             prox_grad[i] = 1
    return prox_val, prox_grad

def trust_region_ssn(
    params: np.ndarray, lbda: float = 1
) -> tuple:
    # https://arxiv.org/pdf/2106.09340
    L = 1
    tau = 0.1 / (lbda**2 * L**2 + 2)
    nu = 0.5 * min(tau, 0.05 * (1 - 0.5 * tau * (0.5 * L**2 * lbda**2 + 1)))
    n_s = 0

    prox_params, D_diagonal = prox_and_grad(params, lbda)
    normal_map_vector = normal_map(params, prox_params, lbda)
    normal_map_norm = jnp.linalg.norm(normal_map_vector)
    delta = min(0.01, 0.5 * jnp.linalg.norm(normal_map_vector))
    H_value = H(
        normal_map_vector, prox_params, tau, lbda
    )

    for _ in range(1000):
        if normal_map_norm < 5e-14:
            return prox_params

        # t = time.time()
        opposite_D_diagonal = np.ones_like(D_diagonal) - D_diagonal
        # logging.info(f"prepare CG1 {time.time()-t}")
        # t = time.time()
        g_vector = jnp.multiply(D_diagonal, normal_map_vector)
        # logging.info(f"prepare CG2 {time.time()-t}")
        # t = time.time()
        hess_D_g = jax.jvp(grad_f_N, (prox_params,), (g_vector,))[1]
        # logging.info(f"prepare CG3 {time.time()-t}")
        # t = time.time()
        normal_map_norm = jnp.linalg.norm(normal_map_vector)
        # logging.info(f"prepare CG4 {time.time()-t}")
        # t = time.time()
        S_g = jnp.multiply(D_diagonal, hess_D_g.T).T  # D^T*B*D*g
        # logging.info(f"prepare CG6 {time.time()-t}")
        # t = time.time()

        def m(q: np.ndarray) -> float:
            D_q = jnp.multiply(D_diagonal, q)
            B_D_q = jax.jvp(grad_f_N, (prox_params,), (D_q,))[1]
            quadratic_part = 0.5 * B_D_q @ q
            linear_part = jnp.matmul(normal_map_vector, jnp.multiply(D_diagonal, q))
            return linear_part + quadratic_part
       
        # logging.info(f"prepare CG7 {time.time()-t}")
        # t = time.time()
        eps = min((jnp.power(normal_map_norm, 2.5), 0.01))
        # logging.info(f"prepare CG8 {time.time()-t}")

        # t = time.time()
        q_bar = steihaug_cg(prox_params, D_diagonal, S_g, g_vector, eps, delta, m)
        # logging.info(f"CG {time.time()-t}")

        # t = time.time()
        D_q_bar = jnp.multiply(D_diagonal, q_bar)
        hess_D_q_bar = jax.jvp(grad_f_N, (prox_params,), (D_q_bar,))[1] # B*D*q_bar
        s_bar = (
            q_bar
            - lbda * (normal_map_vector + hess_D_q_bar)
            - jnp.multiply(opposite_D_diagonal, q_bar)
        )
        s = min(1, delta / jnp.linalg.norm(s_bar)) * s_bar

        params_plus = params + s
        prox_params_plus, D_diagonal_plus = prox_and_grad(params_plus, lbda)
        normal_map_vector_plus = normal_map(params_plus, prox_params_plus, lbda)
        H_value_plus = H(normal_map_vector_plus, prox_params_plus, tau, lbda)
        # logging.info(f"post CG {time.time()-t}")

        # t = time.time()
        a_red = H_value - H_value_plus
        if n_s:
            nu_k = min(
                nu,
                0.001
                * n_s**0.2
                * np.log(n_s) ** 0.4
                * np.linalg.norm(prox_params_plus - params) ** 0.2,
            )
        else:
            nu_k = nu
        p_red = 0.5 * tau * normal_map_norm * min(
            lbda, delta, lbda * normal_map_norm
        ) + nu_k * normal_map_norm * jnp.linalg.norm(prox_params_plus - prox_params) ** 2 / min(
            delta, lbda * normal_map_norm
        )
        rho = a_red / p_red
        # logging.info(f"rho {time.time()-t}")

        # t = time.time()
        if rho < 1e-6:
            delta *= 0.25  # max(0.01, 0.25 * delta)
            choice = "Redc"
        else:
            n_s += 1
            params = params_plus
            normal_map_vector = normal_map_vector_plus
            normal_map_norm = jnp.linalg.norm(normal_map_vector)
            prox_params = prox_params_plus
            D_diagonal = D_diagonal_plus
            H_value = H_value_plus
            if rho < 0.75:
                choice = "Keep"
            else:
                delta = 2 * delta
                choice = "Incr"
        # logging.info(f"update {time.time()-t}")
        logging.info(f"choice: {choice}, delta: {delta}, rho: {rho:.3E}, n_s: {n_s}, normal_map: {np.linalg.norm(normal_map_vector)}, grad: {np.linalg.norm(grad_j_N(prox_params)):.3E} objective {j_N(prox_params):.3E}")

def constraint_finder(q: np.ndarray, p: np.ndarray, delta: float) -> tuple:
    q_q = jnp.matmul(q, q)
    p_p = jnp.matmul(p, p)
    q_p = jnp.matmul(q, p)
    a_negative = (-q_p - jnp.sqrt((q_p) ** 2 - (q_q - delta**2) * p_p)) / (p_p)
    a_positive = (-q_p + jnp.sqrt((q_p) ** 2 - (q_q - delta**2) * p_p)) / (p_p)
    return a_negative, a_positive

def steihaug_cg(
    prox_params: np.ndarray,
    D_diagonal: np.ndarray,
    S_g: np.ndarray,
    g_vector: np.ndarray,
    eps: float,
    delta: float,
    m: Callable,
) -> np.ndarray:
    i = 0
    r = g_vector
    r_r = r @ r
    q = np.zeros_like(g_vector)
    p = -g_vector
    S_p = -S_g
    if jnp.linalg.norm(r) < eps:
        logging.info(
            f"Steihaug1: {i} iterations, delta {delta}, norm {jnp.linalg.norm(q)}, norm r {jnp.linalg.norm(r)}, eps {eps}"
        )
        return q
    while i < len(g_vector):
        # t = time.time()
        # logging.info(f"CG1: {time.time()-t}")
        # t = time.time()
        p_S_p = jnp.matmul(S_p, p)
        # logging.info(f"CG2: {time.time()-t}")
        if p_S_p <= 0:
            a_minus, a_plus = constraint_finder(q, p, delta)
            q_minus = q + a_minus * p
            q_plus = q + a_plus * p
            m_q_minus = m(q_minus)
            m_q_plus = m(q_plus)
            if m_q_minus < m_q_plus:
                logging.info(
                    f"Steihaug2: {i} iterations, delta {delta}, norm {jnp.linalg.norm(q_minus)}, norm r {jnp.linalg.norm(r)}, eps {eps}"
                )
                return q_minus
            else:
                logging.info(
                    f"Steihaug3: {i} iterations, delta {delta}, norm {jnp.linalg.norm(q_plus)}, norm r {jnp.linalg.norm(r)}, eps {eps}"
                )
                return q_plus
        # t = time.time()
        a = r_r / p_S_p
        q_plus = q + a * p
        # logging.info(f"CG3: {time.time()-t}")
        # t = time.time()
        if jnp.linalg.norm(q_plus) > delta:
            a_minus, a_plus = constraint_finder(q, p, delta)
            if a_plus >= 0:
                logging.info(
                    f"Steihaug4: {i} iterations, delta {delta}, norm {jnp.linalg.norm(q+a_plus*p)}, norm r {jnp.linalg.norm(r)}, eps {eps}"
                )
                return q + a_plus * p
            elif a_minus >= 0:
                logging.info(
                    f"Steihaug5: {i} iterations, delta {delta}, norm {jnp.linalg.norm(q+a_minus*p)}, norm r {jnp.linalg.norm(r)}, eps {eps}"
                )
                return q + a_minus * p
            else:
                return
        # logging.info(f"CG4: {time.time()-t}")
        # t = time.time()
        r_plus = r + a * S_p
        r_plus_r_plus = r_plus @ r_plus
        if np.linalg.norm(r_plus) < eps:
            logging.info(
                f"Steihaug6: {i} iterations, delta {delta}, norm {np.linalg.norm(q_plus)}, norm r {np.linalg.norm(r_plus)}, eps {eps}"
            )
            return q_plus
        beta = r_plus_r_plus / r_r
        p_plus = -r_plus + beta * p

        q = q_plus.copy()
        r = r_plus.copy()
        r_r = r_plus_r_plus
        p = p_plus.copy()
        hess_D_p = jax.jvp(grad_f_N, (prox_params,), (jnp.multiply(D_diagonal, p),))[1]
        S_p = jnp.multiply(D_diagonal, hess_D_p.T).T  # D^T*B*D*p
        i += 1
        # logging.info(f"CG5: {time.time()-t}")
    logging.info(
        f"Steihaug7: {i} iterations, delta {delta}, norm {np.linalg.norm(q)}, norm r {np.linalg.norm(r)}, eps {eps}"
    )
    return q

In [34]:
# params = np.array([ 1.81450696,  0.26771097,  0.64023303, -0.62617574,  1.86121196,  0.32289729,
#  -0.5013581,  -0.50221212,  1.35357906,  0.33028574, -0.66661876,  0.6672272,
#   1.04991046,  0.33333333,  0.66666667,  0.66666667,  6.47795428,  0.53598473,
#   0.03417592,  0.04698021, -0.15034678])
params = np.zeros(1000)
params = trust_region_ssn(params, lbda=10)

INFO:root:Steihaug1: 0 iterations, delta 0.01, norm 0.0, norm r 0.0, eps 0.01
INFO:root:choice: Incr, delta: 0.02, rho: 1.999E+00, n_s: 1, normal_map: 1.1417623711534977, grad: 1.154E+00 objective 6.931E-01


INFO:root:Steihaug1: 0 iterations, delta 0.02, norm 0.0, norm r 0.0, eps 0.01
INFO:root:choice: Incr, delta: 0.04, rho: 1.998E+00, n_s: 2, normal_map: 1.1397623711534977, grad: 1.154E+00 objective 6.931E-01
INFO:root:Steihaug1: 0 iterations, delta 0.04, norm 0.0, norm r 0.0, eps 0.01
INFO:root:choice: Incr, delta: 0.08, rho: 1.996E+00, n_s: 3, normal_map: 1.1357623711534977, grad: 1.154E+00 objective 6.931E-01
INFO:root:Steihaug1: 0 iterations, delta 0.08, norm 0.0, norm r 0.0, eps 0.01
INFO:root:choice: Incr, delta: 0.16, rho: 4.132E+02, n_s: 4, normal_map: 1.0844295738818583, grad: 1.088E+00 objective 6.724E-01
INFO:root:Steihaug4: 0 iterations, delta 0.16, norm 0.16, norm r 0.7348002598291813, eps 0.01
INFO:root:choice: Incr, delta: 0.32, rho: 3.249E+02, n_s: 5, normal_map: 1.1087226682756526, grad: 1.097E+00 objective 6.428E-01
INFO:root:Steihaug4: 0 iterations, delta 0.32, norm 0.32, norm r 0.7493591919113547, eps 0.01
INFO:root:choice: Incr, delta: 0.64, rho: 7.535E+01, n_s: 6, n

In [35]:
indices = params != 0
grd = grad_j_N(params)
np.linalg.norm(grd[indices])

7.617412908595647e-17

In [36]:
alpha

0.002

In [37]:
np.max(grd[~indices])

Array(0.00399473, dtype=float64)

In [ ]:
S_matrix = np.array([[ 1.04878306e-01,  6.73990156e-01,  9.31341335e-03, -1.16027736e-01,
   1.18086655e-06,  7.17825416e-05, -1.36522341e-05,  1.36521946e-05,
   2.50953943e-02,  2.96795545e-01, -3.09914467e-02,  1.43279305e-01,
   5.65959722e-02,  1.37703138e+00, -6.67484391e-01,  4.88935027e-01,
   1.28922300e+00],
 [ 6.73990156e-01,  5.13673742e+00,  4.54401345e-01, -1.40808426e+00,
   7.17825416e-05,  3.75959006e-03, -7.64124430e-04,  7.64119665e-04,
   3.06017195e-01,  2.62822349e+00, -2.67565552e-01,  1.39636688e+00,
   5.70400254e-01,  1.02455577e+01, -5.30704796e+00,  3.58324156e+00,
   9.91166301e+00],
 [ 9.31341335e-03,  4.54401345e-01,  1.52566777e+00,  2.83503763e-01,
   1.36521946e-05,  7.64119665e-04, -1.44132657e-04,  1.57835293e-04,
   3.11245004e-02,  2.67790933e-01,  1.22558571e-01,  1.77701802e-01,
   1.54491176e-01,  1.36853162e+00, -6.77648665e-01,  1.33465589e+00,
   7.44766153e-01],
 [-1.16027736e-01, -1.40808426e+00,  2.83503763e-01,  1.06948495e+00,
  -1.36522341e-05, -7.64124430e-04,  1.57836206e-04, -1.44132657e-04,
  -1.21144999e-01, -9.09147818e-01,  1.49607484e-01, -5.29927474e-01,
  -1.54511582e-01, -2.49623113e+00,  1.82228638e+00, -4.28081294e-01,
  -2.17928979e+00],
 [ 1.18086655e-06,  7.17825416e-05,  1.36521946e-05, -1.36522341e-05,
   1.04878306e-01,  6.73990156e-01, -1.16027736e-01,  9.31341335e-03,
   2.50953943e-02,  2.96795545e-01,  1.43279305e-01, -3.09914467e-02,
   5.65959722e-02,  1.37703138e+00,  4.88935027e-01, -6.67484391e-01,
   1.28922300e+00],
 [ 7.17825416e-05,  3.75959006e-03,  7.64119665e-04, -7.64124430e-04,
   6.73990156e-01,  5.13673742e+00, -1.40808426e+00,  4.54401345e-01,
   3.06017195e-01,  2.62822349e+00,  1.39636688e+00, -2.67565552e-01,
   5.70400254e-01,  1.02455577e+01,  3.58324156e+00, -5.30704796e+00,
   9.91166301e+00],
 [-1.36522341e-05, -7.64124430e-04, -1.44132657e-04,  1.57836206e-04,
  -1.16027736e-01, -1.40808426e+00,  1.06948495e+00,  2.83503763e-01,
  -1.21144999e-01, -9.09147818e-01, -5.29927474e-01,  1.49607484e-01,
  -1.54511582e-01, -2.49623113e+00, -4.28081294e-01,  1.82228638e+00,
  -2.17928979e+00],
 [ 1.36521946e-05,  7.64119665e-04,  1.57835293e-04, -1.44132657e-04,
   9.31341335e-03,  4.54401345e-01,  2.83503763e-01,  1.52566777e+00,
   3.11245004e-02,  2.67790933e-01,  1.77701802e-01,  1.22558571e-01,
   1.54491176e-01,  1.36853162e+00,  1.33465589e+00, -6.77648665e-01,
   7.44766153e-01],
 [ 2.50953943e-02,  3.06017195e-01,  3.11245004e-02, -1.21144999e-01,
   2.50953943e-02,  3.06017195e-01, -1.21144999e-01,  3.11245004e-02,
   3.87370504e-01,  1.64417242e+00,  8.33775008e-02,  8.33775008e-02,
   2.85141084e-01,  4.48983766e+00, -1.57192365e+00, -1.57192365e+00,
   4.61307052e+00],
 [ 2.96795545e-01,  2.62822349e+00,  2.67790933e-01, -9.09147818e-01,
   2.96795545e-01,  2.62822349e+00, -9.09147818e-01,  2.67790933e-01,
   1.64417242e+00,  7.49357683e+00,  7.29082777e-01,  7.29082777e-01,
   1.54908793e+00,  2.31508543e+01, -3.62585237e+00, -3.62585237e+00,
   2.50503394e+01],
 [-3.09914467e-02, -2.67565552e-01,  1.22558571e-01,  1.49607484e-01,
   1.43279305e-01,  1.39636688e+00, -5.29927474e-01,  1.77701802e-01,
   8.33775008e-02,  7.29082777e-01,  2.02408485e+00, -3.89286766e-01,
   4.51178428e-01,  2.28898328e+00,  2.55063982e+00, -2.48725309e+00,
   2.38536551e+00],
 [ 1.43279305e-01,  1.39636688e+00,  1.77701802e-01, -5.29927474e-01,
  -3.09914467e-02, -2.67565552e-01,  1.49607484e-01,  1.22558571e-01,
   8.33775008e-02,  7.29082777e-01, -3.89286766e-01,  2.02408485e+00,
   4.51178428e-01,  2.28898328e+00, -2.48725309e+00,  2.55063982e+00,
   2.38536551e+00],
 [ 5.65959722e-02,  5.70400254e-01,  1.54491176e-01, -1.54511582e-01,
   5.65959722e-02,  5.70400254e-01, -1.54511582e-01,  1.54491176e-01,
   2.85141084e-01,  1.54908793e+00,  4.51178428e-01,  4.51178428e-01,
   5.12160322e-01,  5.89689089e+00, -2.09888879e-01, -2.09888879e-01,
   5.97095957e+00],
 [ 1.37703138e+00,  1.02455577e+01,  1.36853162e+00, -2.49623113e+00,
   1.37703138e+00,  1.02455577e+01, -2.49623113e+00,  1.36853162e+00,
   4.48983766e+00,  2.31508543e+01,  2.28898328e+00,  2.28898328e+00,
   5.89689089e+00,  1.03068464e+02, -7.31277333e+00, -7.31277333e+00,
   9.79807873e+01],
 [-6.67484391e-01, -5.30704796e+00, -6.77648665e-01,  1.82228638e+00,
   4.88935027e-01,  3.58324156e+00, -4.28081294e-01,  1.33465589e+00,
  -1.57192365e+00, -3.62585237e+00,  2.55063982e+00, -2.48725309e+00,
  -2.09888879e-01, -7.31277333e+00,  2.76253970e+01, -6.53209081e-01,
  -6.47814211e+00],
 [ 4.88935027e-01,  3.58324156e+00,  1.33465589e+00, -4.28081294e-01,
  -6.67484391e-01, -5.30704796e+00,  1.82228638e+00, -6.77648665e-01,
  -1.57192365e+00, -3.62585237e+00,  2.55063982e+00, -2.48725309e+00,
  -2.09888879e-01, -7.31277333e+00,  2.76253970e+01, -6.53209081e-01,
  -6.47814211e+00],
 [ 1.28922300e+00,  9.91166301e+00,  7.44766153e-01, -2.17928979e+00,
   1.28922300e+00,  9.91166301e+00, -2.17928979e+00,  7.44766153e-01,
   4.61307052e+00,  2.50503394e+01,  2.38536551e+00,  2.38536551e+00,
   5.97095957e+00,  9.79807873e+01, -6.47814211e+00, -6.47814211e+00,
   1.00000000e+02]])
g_vector = np.array([ 1.62087481e-02,  2.76655322e-01, -1.35288550e-03, -5.20239757e-02,
  1.62087481e-02,  2.76655322e-01, -5.20239757e-02, -1.35288550e-03,
  8.12830358e-02,  5.53038453e-01,  5.38640736e-02,  5.38640736e-02,
  8.66640705e-02,  1.60879894e+00, -3.55244756e-01, -3.55244756e-01,
  1.53221690e+00])
eps=0.01
delta=2.2737367544323207e-14
D_diagonal = np.array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
normal_map_vector = np.array([ 1.62087481e-02,  2.76655322e-01, -1.35288550e-03, -5.20239757e-02,
  1.62087481e-02,  2.76655322e-01, -5.20239757e-02, -1.35288550e-03,
  8.12830358e-02,  5.53038453e-01,  5.38640736e-02,  5.38640736e-02,
  8.66640705e-02,  1.60879894e+00, -3.55244756e-01, -3.55244756e-01,
  1.53221690e+00])

## Experiments

In [20]:
exp = NLGCG(target=target, 
           kernel=kernel, 
           g=g,
           f=f,
           grad_f=grad_f,
           hess_f=hess_f,
           grad_f_N=grad_f_N,
           hess_f_N=hess_f_N,
           j=j,
           j_N=j_N,
           p=p,
           grad_p=grad_p,
           hess_p=hess_p,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=10,
           dual_variable_goodness=0.3,
           armijo_constant=1e-1,
           max_inner_loop=25,
           constant_dim=len(target),
           kernel_dim=len(target)
           )

In [ ]:
u, c, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.nlgcg(tol=5e-14, inner_mode="newton", max_radius=0.000001, mode="deterministic")

INFO:root:SSN in 1 dimensions converged in 0 iterations to tolerance 5.000E-14
INFO:root:SSN in 1 dimensions converged in 0 iterations to tolerance 5.000E-14
INFO:root:0: choice: 0, lazy: True, support: 0, epsilon: 9.766E+00, criterion: 5.761E-02, c_raw: 1.0, objective: 1.95315690047590E+01
INFO:root:=============================================================================================
INFO:root:SSN in 11 dimensions converged in 1 iterations to tolerance 5.000E-14
INFO:root:[1e-06, 1e-06, 1e-06, 1e-06, 1e-06, 1e-06, 1e-06, 1e-06, 1e-06, 1e-06]
INFO:root:1, 0: Globalization: NotA, support: 10, c_raw: 1.00E+00, sigma: 0.00E+00, epsilon: 9.77E+00, criterion: 5.76E-02, objective: 1.15341848762528E+01
INFO:root:hess: 0.439807653427124
INFO:root:inv: 0.07648992538452148
INFO:root:norm: 0.021121978759765625
INFO:root:gad_direction: 0.025774717330932617
INFO:root:condition: 0.011987924575805664
INFO:root:condition check 0.007834672927856445
INFO:root:j_N: 4.1961669921875e-05
INFO:root:d

In [ ]:
full_parameters = np.hstack((u.to_matrix().flatten(), c))

In [ ]:
grd = exp.grad_j_N(full_parameters)
np.linalg.norm(grd)

In [ ]:
hss = exp.hess_j_N(full_parameters)
np.linalg.eigvals(hss)

In [ ]:
print(u)

In [ ]:
c

## Plots

In [ ]:
# Dual variable in 1D omega space
p_u = p(u, c)
resolution = 100
a_sigma = np.linspace(0,2,resolution, endpoint=False)
a_omega = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
x, y = np.meshgrid(a_omega,a_omega)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = np.abs(p_u(points)).reshape((resolution,resolution))

plt.contourf(x, y, vals, levels=100);
for omega in u.support:
    plt.plot(omega[0], omega[1], "o", c="black", markersize=5);
plt.colorbar();
# plt.xlim(0,0.5)

In [ ]:
p_u = p(u, c)
p_u(u.support)

In [ ]:
# Projection of the dual variable on an axis
p_u = p(u, c)
resolution = 100
a_sigma = np.linspace(0,1,resolution, endpoint=False)
index = 11
points = np.array([u.support[index]]*resolution)
points[:,0]=a_sigma
vals = np.abs(p_u(points))

plt.plot(a_sigma, vals);
plt.plot([u.support[index][0]], np.abs(p_u(np.array([u.support[index]]))), "o", c="black", markersize=5);

In [ ]:
# Parameterized objective, projected into 2D
point = np.array([1,0.3,0.3,0.3,0.1]) # u.to_matrix().flatten()
resolution = 100
a_1 = np.linspace(0,1,resolution, endpoint=False)
a_2 = np.linspace(0,1,resolution, endpoint=False)
x, y = np.meshgrid(a_1,a_2)
raw_points = np.array(list(zip(x.flatten(), y.flatten())))
points = np.array([point]*len(raw_points))
points[:,2:4] = raw_points

# points = np.hstack((0.02*np.ones((points.shape[0],1)), points))
vals = np.log(jax.vmap(j_N)(points)).reshape((resolution,resolution))

plt.contourf(x, y, vals, levels=100);
plt.colorbar();

In [ ]:
# 1D problem with both true and predicted sources
a = np.arange(0,1,0.01).reshape(-1,1)

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - a,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**(variance_exponent))/(np.sqrt(2*np.pi)**d)
    return outer

true_vals = true_measure.duality_pairing(jax.vmap(plot_kernel)) # .reshape((100,100))
pred_vals = u.duality_pairing(jax.vmap(plot_kernel)) + c*np.ones(len(a))

plt.plot(a, true_vals, c="blue")
plt.plot(a, pred_vals, c="red");
plt.xlim(0,1);
plt.ylim(-1, 1);
signs = np.sign(true_weights)
for i, x in enumerate(true_sources):
    size = signs[i]*(0.01+np.sqrt(np.abs(true_weights[i]))*0.3)+0.5
    plt.axvline(x=x[1], ymin=min(0.5,size)+0.01,ymax=max(0.5, size)-0.01, linewidth=5, c="blue", alpha=sp.special.expit(np.log(np.sqrt(x[0]*5))));
signs = np.sign(u.coefficients)
for i, x in enumerate(u.support):
    size = signs[i]*(0.01+np.sqrt(np.abs(u.coefficients[i]))*0.3)+0.5
    plt.axvline(x=x[1], ymin=min(0.5,size)+0.01,ymax=max(0.5,size)-0.01, linewidth=5, c="red", alpha=sp.special.expit(np.log(np.sqrt(x[0]*5))));

In [ ]:
# 1D problem with a true function and predicted sources
resolution = 1000
a = np.linspace(omega_space[0][0],omega_space[0][1],resolution,endpoint=False).reshape(-1,1)

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - a,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**(variance_exponent))/(np.sqrt(2*np.pi)**d)
    return outer

true_vals = true_function(a)
pred_vals = u.duality_pairing(jax.vmap(plot_kernel)) + c*np.ones(len(a))

plt.plot(a, true_vals, c="blue")
plt.plot(a, pred_vals, c="red");
plt.xlim(omega_space[0][0],omega_space[0][1]);
signs = np.sign(u.coefficients)
for i, x in enumerate(u.support):
    if signs[i]>0:
        color = "blue"
    else:
        color = "red"
    size = (0.01+np.sqrt(np.abs(u.coefficients[i]))*0.5)
    plt.axvline(x=x[1], ymin=0.01,ymax=size-0.01, linewidth=5, c=color, alpha=sp.special.expit(np.log(np.sqrt(x[0]*5))));

In [ ]:
# 2D problem with both true and predicted sources
a = np.arange(0,1,0.01).reshape(-1,1)
B, C = np.meshgrid(a,a)
x = np.array(list(zip(B.flatten(), C.flatten())))

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - x,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**variance_exponent)/(np.sqrt(2*np.pi)**d)
    return outer

true_vals = true_measure.duality_pairing(jax.vmap(plot_kernel)).reshape((100,100))
pred_vals = u.duality_pairing(jax.vmap(plot_kernel)).reshape((100,100))

fig, axes = plt.subplots()
plt.contourf(B, C, true_vals, levels=100);
plt.colorbar();
plt.xlim(0, 1);
plt.ylim(0, 1);
for i, x in enumerate(true_sources):
    if true_weights[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(true_weights[i])*10
    plt.plot([x[1]], [x[2]], "P", c=color, markersize=size, alpha=0.5);
    circle = plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5);
    axes.add_artist(circle);
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i])*10
    plt.plot([x[1]], [x[2]], "o", c=color, markersize=size, alpha=0.5);
    circle = plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5, linestyle="--");
    axes.add_artist(circle);

In [ ]:
# 2D problem with a true function and predicted sources
resolution = 100
a_1 = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
a_2 = np.linspace(omega_space[1][0],omega_space[1][1],resolution, endpoint=False)
x, y = np.meshgrid(a_1,a_2)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = true_function(points).reshape((resolution,resolution))

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - points,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**variance_exponent)/(np.sqrt(2*np.pi)**d)
    return outer

pred_vals = u.duality_pairing(jax.vmap(plot_kernel)).reshape((resolution,resolution)) +c*np.ones((resolution, resolution))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15,4))
contour1 = ax1.contourf(x, y, vals, levels=100)
fig.colorbar(contour1, ax=ax1)
contour2 = ax2.contourf(x, y, pred_vals, levels=100)
fig.colorbar(contour2, ax=ax2)
contour3 = ax3.contourf(x, y, np.abs(pred_vals-vals), levels=100)
fig.colorbar(contour3, ax=ax3)
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i])
    ax1.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
    ax2.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
    ax3.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
ax1.set_xlim(omega_space[0][0],omega_space[0][1])
ax1.set_ylim(omega_space[1][0],omega_space[1][1])
ax2.set_xlim(omega_space[0][0],omega_space[0][1])
ax2.set_ylim(omega_space[1][0],omega_space[1][1])
ax3.set_xlim(omega_space[0][0],omega_space[0][1])
ax3.set_ylim(omega_space[1][0],omega_space[1][1])

In [ ]:
# Higher-dimensional problem
resolution = 20
a_1 = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
a_2 = np.linspace(omega_space[1][0],omega_space[1][1],resolution, endpoint=False)
a_3 = np.linspace(omega_space[2][0],omega_space[2][1],resolution, endpoint=False)
a_4 = np.linspace(omega_space[3][0],omega_space[3][1],resolution, endpoint=False)
w, x, y, z = np.meshgrid(a_1,a_2,a_3,a_4)
points = np.array(list(zip(w.flatten(), x.flatten(), y.flatten(), z.flatten())))
vals = true_function(points)

def plot_kernel(omega: np.ndarray):
    sigma = omega[0]
    x_omega = omega[1:]
    inner = -(jnp.linalg.norm(x_omega - points,axis=1)**2)/(2*sigma**2)
    outer = (jnp.exp(inner)*sigma**variance_exponent)/(np.sqrt(2*np.pi)**d)
    return outer

pred_vals = u.duality_pairing(jax.vmap(plot_kernel)) +c*np.ones(points.shape[0])

err = np.abs(vals-pred_vals)
print(f"L2 error: {np.linalg.norm(err)}, Linf error: {np.max(err)}, average error: {np.mean(err)}")

In [ ]:
# Convergence in iterations with shaded inner loops
intervals = []
current_inner = False
for ind, i in enumerate(inner_loop):
    if i:
        if not current_inner:
            start = ind-0.5
            current_inner = True
    else:
        if current_inner:
            end = ind-0.5
            intervals.append((start,end))
            current_inner = False
if inner_loop[-1]:
    end = len(inner_loop)-0.5
    intervals.append((start,end))

residuals = np.array(objective_values) - np.min(objective_values)
plt.figure(figsize=(11.25,5))
plt.semilogy(np.array(range(len(residuals))), residuals, linestyle="-.", color="green");
for interval in intervals:
    plt.fill_between(interval, 0, 60, hatch="/", color="gray", alpha=0.2);
plt.ylim(1e-17, 60);
# plt.xlim(2500, 3000);
plt.ylabel("Objective residual");
plt.xlabel("Total iterations");
plt.show()

# PDEs

## Generate Data and Define Functions

In [ ]:
logging.getLogger().setLevel(logging.INFO) # Supress logging

In [ ]:
sigma_space = np.array([0,1])
omega_space = np.array([[-1,1], [-1,1]])
Omega = np.vstack((sigma_space, omega_space))
d = omega_space.shape[0]
variance_exponent = 2.01
alpha = 1e-2
lambda_ = 1000
observation_resolution = 20

@jax.jit
def true_function(x: np.ndarray):
    return jnp.tanh(4*(0.3-jnp.linalg.norm(x-np.array([0.300000000000000001,0.3000000000000001]))))+jnp.tanh(12*(0.15-jnp.linalg.norm(x+np.array([0.300000000000001,0.3000000000000001]))))+2
grad_true_function = jax.jit(jax.grad(true_function))
hess_true_function = jax.jit(jax.hessian(true_function))
true_function = jax.vmap(true_function)
grad_true_function = jax.vmap(grad_true_function)
hess_true_function = jax.vmap(hess_true_function)

int_function = true_function
grad_int_function = grad_true_function
hess_int_function = hess_true_function
laplacian_int_function = lambda omega: jnp.trace(hess_int_function(omega),axis1=1, axis2=2)
bound_function = true_function

In [ ]:
observations_raw = (np.array(np.meshgrid(
                    *(
                        np.linspace(bound[0], bound[1], observation_resolution, endpoint=True)
                        for bound in omega_space
                    ))
            ).reshape(len(omega_space), -1).T)
int_observations = np.array([obs for obs in observations_raw if all(obs!=omega_space[0][0]) and all(obs!=omega_space[0][1])])
bound_observations = np.array([obs for obs in observations_raw if any(obs==omega_space[0][0]) or any(obs==omega_space[0][1])])

In [ ]:
int_target = int_function(int_observations)**3-laplacian_int_function(int_observations)
bound_target = np.sqrt(lambda_) * bound_function(bound_observations)
target = np.hstack((int_target, bound_target))
observations = np.vstack((int_observations, bound_observations))

constant_dim = len(target)
kernel_dim = len(target)+len(int_target)

In [ ]:
@jax.jit
def adjusted_sigmoid(x):
    return 2*jnp.exp(2*x)/(1+jnp.exp(2*x))-1

@jax.jit
def raw_kernel(omega: np.ndarray, x: np.ndarray):
    # Both inputs 1D
    sigma = omega[0]
    x_omega = omega[1:]
    variance_factor = adjusted_sigmoid(sigma**variance_exponent)
    inner = -(jnp.sum((x_omega - x)**2))/(2*sigma**2)
    outer = (jnp.exp(inner)*variance_factor)/(np.sqrt(2*np.pi)**d)
    return outer

hess_raw_kernel = jax.jit(jax.hessian(raw_kernel, argnums=1))
raw_kernel = jax.vmap(raw_kernel, (None, 0), 0)
hess_raw_kernel = jax.vmap(hess_raw_kernel, (None, 0), 0)

@jax.jit
def singleton_kernel(omega: np.ndarray):
    # function, laplacian
    return jnp.hstack([raw_kernel(omega, observations),jnp.trace(hess_raw_kernel(omega, int_observations),axis1=1, axis2=2)])

grad_kernel = jax.jit(jax.jacobian(singleton_kernel))
hess_kernel = jax.jit(jax.hessian(singleton_kernel))
_ = grad_kernel(np.ones(len(Omega)))
_ = hess_kernel(np.ones(len(Omega)))

kernel = jax.vmap(singleton_kernel)
grad_kernel = jax.vmap(grad_kernel)
hess_kernel = jax.vmap(hess_kernel)

In [ ]:
g = jax.jit(lambda w: alpha * jnp.linalg.norm(w, ord=1))
grad_g = jax.jit(jax.grad(g))
_ = grad_g(jnp.ones(1))

@jax.jit
def r(function_and_laplacian: np.ndarray) -> np.ndarray:
    u = function_and_laplacian[:len(target)] # shape (len(target),)
    laplacian = function_and_laplacian[len(target):] # shape (len(int_target),)
    to_return = jnp.hstack((u[:len(int_target)]**3 - laplacian, np.sqrt(lambda_) * u[len(int_target):]))
    return to_return

f = jax.jit(lambda y: 0.5 * jnp.sum((r(y)-target)**2)*4/len(target))
grad_f = jax.jit(jax.grad(f))
hess_f = jax.jit(jax.hessian(f))
_ = grad_f(jnp.ones(2*len(int_target)+len(bound_target)))
_ = hess_f(jnp.ones(2*len(int_target)+len(bound_target)))

j = lambda u, c: f(u.duality_pairing(kernel, kernel_dim)+np.hstack((c*np.ones(constant_dim), np.zeros(kernel_dim-constant_dim)))) + g(u.coefficients)

In [ ]:
# def p(u: Measure, c: float) -> Callable:
#     inner = -grad_f(u.duality_pairing(kernel, kernel_dim)+np.hstack((c*np.ones(constant_dim), np.zeros(kernel_dim-constant_dim))))
#     return lambda omega: kernel(omega) @ inner

# def grad_p(u: Measure, c: float) -> Callable:
#     inner = -grad_f(u.duality_pairing(kernel, kernel_dim)+np.hstack((c*np.ones(constant_dim), np.zeros(kernel_dim-constant_dim))))
#     return lambda omega: np.tensordot(grad_kernel(omega), inner, axes=([1,0]))

# def hess_p(u: Measure, c: float) -> Callable:
#     inner = -grad_f(u.duality_pairing(kernel, kernel_dim)+np.hstack((c*np.ones(constant_dim), np.zeros(kernel_dim-constant_dim))))
#     return lambda omega: np.tensordot(hess_kernel(omega), inner, axes=([1,0]))

In [ ]:
@jax.jit
def p_raw(parameters: np.ndarray, c: float, omega: np.ndarray):
    coefficients = parameters[:,0]
    support = parameters[:,1:]
    Ku = jnp.tensordot(kernel(support), coefficients, axes=([0], [0]))
    constant_term = jnp.hstack((c*jnp.ones(constant_dim), jnp.zeros(kernel_dim-constant_dim)))
    return singleton_kernel(omega) @ -grad_f(Ku+constant_term)

grad_p_raw = jax.jit(jax.grad(p_raw, argnums=2))
hess_p_raw = jax.jit(jax.hessian(p_raw, argnums=2))

p_raw = jax.jit(jax.vmap(p_raw, in_axes=(None, None, 0), out_axes=0))
grad_p_raw = jax.jit(jax.vmap(grad_p_raw, in_axes=(None, None, 0), out_axes=0))
hess_p_raw = jax.jit(jax.vmap(hess_p_raw, in_axes=(None, None, 0), out_axes=0))

p = lambda u, c: lambda omega: p_raw(u.to_matrix(len(Omega)), c, omega)
grad_p = lambda u, c: lambda omega: grad_p_raw(u.to_matrix(len(Omega)), c, omega)
hess_p = lambda u, c: lambda omega: hess_p_raw(u.to_matrix(len(Omega)), c, omega)

In [ ]:
# Parameterized versions of f and j
@jax.jit
def f_N(raw_input: np.ndarray) -> float:
    constant = raw_input[-1]
    input = raw_input[:-1].reshape(-1, d+2)
    weights = input[:,0]
    omega = input[:,1:]
    return f(kernel(omega).T@weights+constant*jnp.ones(target.shape))

@jax.jit
def j_N(raw_input: np.ndarray) -> float:
    input = raw_input[:-1].reshape(-1, d+2)
    weights = input[:,0]
    return f_N(raw_input) + g(weights)

grad_f_N = jax.jit(jax.grad(f_N))
hess_f_N = jax.jit(jax.hessian(f_N))
grad_j_N = jax.jit(jax.grad(j_N))
hess_j_N = jax.jit(jax.hessian(j_N))

## Experiments

In [ ]:
exp = NLGCG(target=target, 
           kernel=kernel, 
           g=g,
           f=f,
           grad_f=grad_f,
           hess_f=hess_f,
           j=j,
           j_N=j_N,
           p=p,
           grad_p=grad_p,
           hess_p=hess_p,
           grad_j_N=grad_j_N,
           hess_j_N=hess_j_N,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=4,
           dual_variable_goodness=0.75,
           armijo_constant=1e-1,
           max_inner_loop=10,
           constant_dim=constant_dim,
           kernel_dim=kernel_dim,
           grad_f_N=grad_f_N,
           hess_f_N=hess_f_N,
           )

In [ ]:
u = pickle.load(open("u.pkl", "rb")) # 1e-2
c = pickle.load(open("c.pkl", "rb"))

In [ ]:
u, c, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.nlgcg(u_0=u,c_0=c,inner_mode="newton",tol=5e-14, max_radius=0.001, mode="deterministic")

In [ ]:
# pickle.dump(u, open("u.pkl", "wb"))
# pickle.dump(c, open("c.pkl", "wb"))

In [ ]:
print(u)

In [ ]:
c

In [ ]:
full_parameters = np.hstack((u.to_matrix().flatten(), c))

In [ ]:
grd = exp.grad_j_N(full_parameters)
np.linalg.norm(grd)

In [ ]:
hss = exp.hess_j_N(full_parameters)
np.linalg.eigvals(hss)

## Plots

In [ ]:
parameters, u_ks, radii = exp.local_merging_update_radii(u, c)
radii

In [ ]:
# Projection of the dual variable on an axis
p_u = p(u, c)
resolution = 100
a_sigma = np.linspace(0,1,resolution, endpoint=False)
index = -26
points = np.array([u.support[index]]*resolution)
points[:,0]=a_sigma
vals = np.abs(p_u(points))

plt.plot(a_sigma, vals);
plt.plot([u.support[index][0]], np.abs(p_u(np.array([u.support[index]]))), "o", c="black", markersize=5);

In [ ]:
# 2D problem with a true function and predicted sources
resolution = 100
a_1 = np.linspace(omega_space[0][0],omega_space[0][1],resolution, endpoint=False)
a_2 = np.linspace(omega_space[1][0],omega_space[1][1],resolution, endpoint=False)
x, y = np.meshgrid(a_1,a_2)
points = np.array(list(zip(x.flatten(), y.flatten())))
vals = int_function(points).reshape((resolution,resolution))

@jax.jit
def plot_kernel(omega: np.ndarray):
    return raw_kernel(omega, points)
plot_kernel = jax.vmap(plot_kernel)

pred_vals = u.duality_pairing(plot_kernel).reshape((resolution,resolution)) +c*np.ones((resolution, resolution))

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15,4))
contour1 = ax1.contourf(x, y, vals, levels=100)
fig.colorbar(contour1, ax=ax1)
contour2 = ax2.contourf(x, y, pred_vals, levels=100)
fig.colorbar(contour2, ax=ax2)
contour3 = ax3.contourf(x, y, np.abs(pred_vals-vals), levels=100)
fig.colorbar(contour3, ax=ax3)
for i, x in enumerate(u.support):
    if u.coefficients[i] < 0:
        color = "b"
    else:
        color = "r"
    size = abs(u.coefficients[i])
    ax1.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
    ax2.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
    ax3.add_patch(plt.Circle((x[1], x[2]), radius=x[0], color=color, fill=False, alpha=0.5))
ax1.set_xlim(omega_space[0][0],omega_space[0][1])
ax1.set_ylim(omega_space[1][0],omega_space[1][1])
ax2.set_xlim(omega_space[0][0],omega_space[0][1])
ax2.set_ylim(omega_space[1][0],omega_space[1][1])
ax3.set_xlim(omega_space[0][0],omega_space[0][1])
ax3.set_ylim(omega_space[1][0],omega_space[1][1])

# Function Approximation (Sigmoid Shallow NN)

## Generate Data and Define Functions

In [ ]:
alpha = 1e-6

In [ ]:
# Function approximation
observation_size = 100
observation_space = np.array([[0,1], [0,1]])
Omega = np.array([[0,1], [0,1], [0,1]])
true_function = lambda x: np.sin(10*(x[0]**2+x[1]**2))

np.random.seed(49)
columns = []
for bounds in observation_space:
    columns.append(
        np.random.sample((observation_size, 1)) * (bounds[1] - bounds[0]) + bounds[0]
    )
observations = np.concatenate(columns, axis=1)
observations_bias = np.append(np.ones((observations.shape[0],1)),observations,axis=1)
target = np.array([true_function(x) for x in observations])

In [ ]:
activation = lambda omega: 1/(1+np.exp(omega)) # sigmoid
def kappa(x):
    # input is a 2D array of shape (number of points, dimension of observation space +1)
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    linear_operation = x@(observations_bias.T)
    activated = activation(linear_operation)
    return activated # shape (len(x), len(observations))

In [ ]:
def activation_grad(omega: np.ndarray) -> np.ndarray:
    sigmoid = activation(omega)
    return sigmoid * (1 - sigmoid)  # Derivative of the sigmoid function

def grad_kappa(x):
    # input is a 2D array of shape (number of points, dimension of observation space +1)
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    linear_operation = x@(observations_bias.T)
    activated = activation_grad(linear_operation) # shape=(len(x), len(observations))
    inner_derivative = np.repeat(np.expand_dims(observations_bias,axis=0),len(x),axis=0) # shape=(len(x), len(observations), Omega.shape[0])
    chain_rule = np.multiply(activated.reshape((len(x),len(observations),1)), inner_derivative)
    return chain_rule # shape (len(x), len(observations), Omega.shape[0])

In [ ]:
def activation_hess(omega: np.ndarray) -> np.ndarray:
    sigmoid = activation(omega)
    return sigmoid * (1 - sigmoid)*(1-2*sigmoid)  # Derivative of the sigmoid function

def hess_kappa(x):
    # Input is 2D array of shape (number of points, dimension of observation space +1)
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    linear_operation = x@(observations_bias.T)
    activated = activation_hess(linear_operation)
    inner_second_derivative_raw = np.array([np.outer(vect, vect) for vect in observations_bias]) # shape (len(observations),Omega.shape[0],Omega.shape[0])
    inner_second_derivative = np.repeat(np.expand_dims(inner_second_derivative_raw,axis=0),len(x),axis=0) # shape (len(x),len(observations),Omega.shape[0],Omega.shape[0])
    chain_rule = np.multiply(activated.reshape((len(x),len(observations),1,1)), inner_second_derivative)
    return chain_rule # shape (len(x), len(observations), Omega.shape[0], Omega.shape[0])

In [ ]:
g = lambda u: alpha * np.linalg.norm(u, ord=1)
f = lambda u: 0.5 * np.linalg.norm(u.duality_pairing(kappa) - target) ** 2

In [ ]:
def p_raw(u):
    Ku = u.duality_pairing(kappa)
    inner = Ku-target
    return lambda x: -kappa(x) @ inner

p = lambda u: p_raw(u)

In [ ]:
def grad_P_raw(u):
    p_u = p_raw(u)
    inner = target-u.duality_pairing(kappa)
    return lambda x: np.sign(p_u(x)).reshape(-1,1)*np.tensordot(grad_kappa(x), inner, axes=([1,0]))

grad_P = lambda u: grad_P_raw(u)

In [ ]:
def grad_P_raw_sphere(u):
    p_u = p_raw(u)
    inner = target-u.duality_pairing(kappa)
    def grad_P_x(x):
        unprojected = np.sign(p_u(x)).reshape(-1,1)*np.tensordot(grad_kappa(x), inner, axes=([1,0]))
        to_return = np.zeros(unprojected.shape)
        for i, (grad ,position) in enumerate(zip(unprojected, x)):
            # Project onto the ball
            projection = np.eye(len(position)) - np.outer(position, position)
            to_return[i] = projection@grad
        return to_return
    return grad_P_x

grad_P_sphere = lambda u: grad_P_raw_sphere(u)

In [ ]:
def hess_P_raw(u):
    p_u = p_raw(u)
    inner = target-u.duality_pairing(kappa)
    return lambda x: np.sign(p_u(x)).reshape(-1,1,1)*np.tensordot(hess_kappa(x),inner,axes=([1,0]))

hess_P = lambda u: hess_P_raw(u)

In [ ]:
def hess_P_raw_sphere(u):
    p_u = p_raw(u)
    grad_u = grad_P(u)
    inner = target-u.duality_pairing(kappa)
    def hess_P_x(x):
        grads = grad_u(x)
        unprojected = np.sign(p_u(x)).reshape(-1,1,1)*np.tensordot(hess_kappa(x),inner,axes=([1,0]))
        to_return = np.zeros(unprojected.shape)
        for i, (grad, hess, position) in enumerate(zip(grads, unprojected, x)):
            # P_x(hess-position.T*grad*I)
            projection = np.eye(len(position)) - np.outer(position, position)
            identity_factor = position@grad
            to_return[i] = projection@(hess-identity_factor*np.eye(len(position)))@projection
        return to_return
    return hess_P_x

hess_P_sphere = lambda u: hess_P_raw_sphere(u)

In [ ]:
def grad_j(positions, coefs):
    K_matrix = kappa(positions)
    grad_F = (K_matrix.T@coefs).flatten() - target
    nabla_x = coefs.reshape(-1,1)*np.tensordot(grad_kappa(positions), grad_F, axes=([1,0]))
    nabla_u = np.dot(K_matrix, grad_F) + alpha * np.sign(coefs)
    return np.append(nabla_x.flatten(), nabla_u, axis=0).flatten()

In [ ]:
def grad_j_sphere(positions, coefs):
    K_matrix = kappa(positions)
    grad_F = (K_matrix.T@coefs).flatten() - target
    nabla_x = coefs.reshape(-1,1)*np.tensordot(grad_kappa(positions), grad_F, axes=([1,0]))
    for i, (grad ,position) in enumerate(zip(nabla_x, positions)):
        # Project onto the ball
        nabla_x[i] = (np.eye(len(position))-np.outer(position, position))@grad
    nabla_u = np.dot(K_matrix, grad_F) + alpha * np.sign(coefs)
    return np.append(nabla_x.flatten(), nabla_u, axis=0).flatten()

In [ ]:
def hess_j(positions, coefs):
    kappa_values = kappa(positions)
    grad_kappa_values = grad_kappa(positions)
    hess_kappa_values = hess_kappa(positions)
    matrix_dimension = len(positions)*Omega.shape[0] + len(coefs)
    hesse_matrix = np.zeros((matrix_dimension, matrix_dimension))
    step = Omega.shape[0]
    coefs_delay = step*len(positions)
    inner = (kappa_values.T@coefs).flatten() - target
    for i in range(len(positions)):
        # nabla_{x_i,x_j}
        for j in range(len(positions)):
            if j<i:
                continue
            block = coefs[i]*coefs[j]*np.matmul(grad_kappa_values[i].T, grad_kappa_values[j])
            if i==j:
                block += coefs[i]*np.tensordot(hess_kappa_values[i],inner,axes=([0,0]))
            hesse_matrix[i*step:(i+1)*step, j*step:(j+1)*step] = block
            hesse_matrix[j*step:(j+1)*step, i*step:(i+1)*step] = block.T
        # nabla_{x_i,u_j}
        for j in range(len(coefs)):
            block = coefs[i]*np.matmul(grad_kappa_values[i].T, kappa_values[j])
            if i == j:
                block += np.matmul(grad_kappa_values[i].T, inner)
            hesse_matrix[i*step:(i+1)*step, coefs_delay+j] = block
            hesse_matrix[coefs_delay+j, i*step:(i+1)*step] = block.T
    for i in range(len(coefs)):
        # nabla_{u_i,u_j}
        for j in range(len(coefs)):
            if j<i:
                continue
            block = np.dot(kappa_values[i], kappa_values[j])
            hesse_matrix[coefs_delay+i,coefs_delay+j] = block
            hesse_matrix[coefs_delay+j,coefs_delay+i] = block
    return hesse_matrix

In [ ]:
def hess_j_sphere(positions, coefs):
    hessian = hess_j(positions, coefs)
    gradient = grad_j(positions, coefs)
    projection = np.eye(hessian.shape[0])
    inner_matrix = np.eye(hessian.shape[0])
    for i, position in enumerate(positions):
        projection_part = np.eye(len(position))-np.outer(position, position)
        inner_matrix_part = (position@gradient[i*len(position):(i+1)*len(position)])*np.eye(len(position))
        projection[i*len(position):(i+1)*len(position), i*len(position):(i+1)*len(position)] = projection_part
        inner_matrix[i*len(position):(i+1)*len(position), i*len(position):(i+1)*len(position)] = inner_matrix_part
    hesse_system = projection@(hessian-inner_matrix)@projection
    return hesse_system

## Experiments

In [ ]:
exp = NLGCG(target=target, 
           kappa=kappa, 
           g=g, 
           f=f,
           p=p,
           grad_P=grad_P_sphere,
           hess_P=hess_P_sphere,
           grad_j=grad_j_sphere,
           hess_j=hess_j_sphere,
           alpha=alpha,
           Omega=Omega,
           global_search_resolution=100,
           dual_variable_goodness=0.3,
           projection="sphere"
           )

In [ ]:
u, times, supports, inner_loop, lgcg_lazy, lgcg_total, objective_values, dropped_tot, epsilons = exp.nlgcg(tol=1e-12, max_radius=0.1)

## Plots

In [ ]:
a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
X = np.array(list(zip(B.flatten(), D.flatten())))
true_vals = np.array([true_function(x) for x in X]).reshape((100,100))

plt.contourf(B, D, true_vals, levels=100);
plt.colorbar();
plt.xlim(Omega[0][0], Omega[0][1]);
plt.ylim(Omega[1][0], Omega[1][1]);

In [ ]:
def predicted(x):
    # Input is 2D array of shape (number of points, Omega dimension)
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    x_bias = np.append(np.ones((x.shape[0],1)),x,axis=1)
    activated = activation(x_bias@u.support.T) # shape (len(x), len(observations))
    output = np.dot(activated, u.coefficients) # shape (len(x),)
    return output

a = np.arange(0,1,0.01)
B, D = np.meshgrid(a,a)
x = np.array(list(zip(B.flatten(), D.flatten())))
pred_vals = predicted(x).reshape((100,100))
error = true_vals - pred_vals

print(f"L2 error: {np.linalg.norm(error)/np.sqrt(len(x)):.3E}, Linf error: {np.max(np.abs(error)):.3E}")

plt.contourf(B, D, np.abs(error), levels=100);
plt.colorbar();
for i, x in enumerate(observations):
    plt.plot([x[0]], [x[1]], "P", c="r", markersize=4);